In [1]:
import requests
import pandas as pd
import plotly.express as px

# Approximate bounding box for Franklin County, Ohio 
# south, west, north, east (lat/lon)
bbox = (39.7, -83.3, 40.2, -82.7)

query = f"""
[out:json][timeout:900];
// Get all traffic lights in bounding box
(
  node["highway"="traffic_signals"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
  node["crossing"="traffic_signals"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
  node["traffic_signals"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
  way["highway"="traffic_signals"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
  way["crossing"="traffic_signals"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
  relation["traffic_signals"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});
);
out center tags;
"""

url = "https://overpass-api.de/api/interpreter"
print("Querying Overpass API (bounding box for Franklin County, OH)...")
response = requests.post(url, data={'data': query})
response.raise_for_status()
data = response.json()

# Parse results
records = []
for el in data['elements']:
    lat = el.get('lat') or el.get('center', {}).get('lat')
    lon = el.get('lon') or el.get('center', {}).get('lon')
    if lat is None or lon is None:
        continue
    tags = el.get('tags', {})
    records.append({
        'id': el['id'],
        'type': el['type'],
        'lat': lat,
        'lon': lon,
        'name': tags.get('name'),
        'direction': tags.get('traffic_signals:direction'),
        'operator': tags.get('operator'),
    })

df = pd.DataFrame(records)
df.to_csv("Data/franklin_county_traffic_lights.csv", index=False)



Querying Overpass API (bounding box for Franklin County, OH)...


In [2]:
import geopandas as gpd
gpkg_path = "Data/fastfood_location_name_amenity.gpkg"

fastfood = gpd.read_file(gpkg_path)
fastfood = fastfood.to_crs(epsg=4326)  # ensure WGS84 lat/lon
# Convert non-point geometries to centroids
if not all(fastfood.geometry.geom_type == "Point"):
    fastfood["geometry"] = fastfood.geometry.centroid

fastfood["lat"] = fastfood.geometry.y
fastfood["lon"] = fastfood.geometry.x

C:\Users\boie.2\AppData\Local\Temp\ipykernel_29892\823791414.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  fastfood["geometry"] = fastfood.geometry.centroid


In [10]:
# for comparison, load the pizza locations

pizza_path = "Data/pizza_location_name_cuisine.gpkg"

pizza = gpd.read_file(pizza_path)
pizza = pizza.to_crs(epsg=4326)  # ensure WGS84 lat/lon
# Convert non-point geometries to centroids
if not all(pizza.geometry.geom_type == "Point"):
    pizza["geometry"] = pizza.geometry.centroid

pizza["lat"] = pizza.geometry.y
pizza["lon"] = pizza.geometry.x

C:\Users\boie.2\AppData\Local\Temp\ipykernel_29892\200508360.py:9: UserWarning:

Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.




In [16]:
df

,id,type,lat,lon,name,direction,operator
0,197277038,node,39.884358,-82.759023,None,None,None
1,197277113,node,39.904565,-82.773159,None,both,None
2,197277134,node,39.910027,-82.778501,None,None,None
3,197277145,node,39.912295,-82.779316,None,None,None
4,197277148,node,39.914813,-82.779896,None,both,None
...,...,...,...,...,...,...,...
3519,1445273685,way,40.148490,-82.939007,None,None,None
3520,1445273700,way,40.145915,-82.939134,None,None,None
3521,1445273701,way,40.146082,-82.939309,None,None,None
3522,1445273702,way,40.146371,-82.939023,None,None,None


In [17]:
fig = px.scatter_mapbox(
    df, # the traffic lights (red)
    lat="lat",
    lon="lon",
    hover_name="id",
    #hover_data=["type", "direction", "operator"],
    color_discrete_sequence=["red"],
    zoom=10,
    height=700,
)

# Add fast-food locations (blue)
fig.add_scattermapbox(
    lat=fastfood["lat"],
    lon=fastfood["lon"],
    mode="markers",
    marker=dict(size=8, color="blue", opacity=0.6),
    name="Fast-food",
    hovertext=fastfood.get("name", "Fastfood"),
)

# Add pizza locations (green)
fig.add_scattermapbox(
    lat=pizza["lat"],
    lon=pizza["lon"],
    mode="markers",
    marker=dict(size=8, color="green", opacity=0.6),
    name="Pizza",
    hovertext=pizza.get("name", "Pizza"),
)


fig.update_layout(
    mapbox_style="carto-positron",
    title="Traffic Lights (red), Fastfood Locations (blue), Pizza Restaurant Locations (green) \n Franklin County, Ohio",
    margin={"r":0,"t":40,"l":0,"b":0},
)

In [4]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Load fastfood places from GeoPackage
fastfood_gdf = gpd.read_file("Data/fastfood_location_name_amenity.gpkg").to_crs(epsg=4326)
if not all(fastfood_gdf.geometry.geom_type == "Point"):
    print("Non-point geometries detected — converting to centroids.")
    fastfood_gdf["geometry"] = fastfood_gdf.geometry.centroid

# Traffic lights from the CSV generated earlier
traffic_df = pd.read_csv("Data/franklin_county_traffic_lights.csv")
traffic_gdf = gpd.GeoDataFrame(
    traffic_df,
    geometry=gpd.points_from_xy(traffic_df["lon"], traffic_df["lat"]),
    crs="EPSG:4326"
)

# Project to local metric CRS (UTM 17N for Ohio)
fastfood_gdf = fastfood_gdf.to_crs(epsg=26917)
fastfood_gdf = fastfood_gdf.to_crs(epsg=4326)

min_lat, max_lat = 39.75, 40.15
min_lon, max_lon = -83.25, -82.75

fastfood_franklin = fastfood_gdf[
    (fastfood_gdf.geometry.y > min_lat) &
    (fastfood_gdf.geometry.y < max_lat) &
    (fastfood_gdf.geometry.x > min_lon) &
    (fastfood_gdf.geometry.x < max_lon)
]
fastfood_gdf = fastfood_franklin

nearest_indices = []
nearest_distances = []

for ff_geom in fastfood_gdf.geometry:
    # Find nearest traffic light index
    distances = traffic_gdf.geometry.distance(ff_geom)
    nearest_idx = distances.idxmin()
    nearest_indices.append(nearest_idx)
    nearest_distances.append(distances[nearest_idx])

# Build DataFrame
nearest_df = pd.DataFrame({
    "fastfood_name": fastfood_gdf.get("name_normalized", pd.Series([None]*len(fastfood_gdf))),
    "fastfood_lat": fastfood_gdf.to_crs(epsg=4326).geometry.y,
    "fastfood_lon": fastfood_gdf.to_crs(epsg=4326).geometry.x,
    "nearest_light_lat": traffic_gdf.to_crs(epsg=4326).geometry.iloc[nearest_indices].y.values,
    "nearest_light_lon": traffic_gdf.to_crs(epsg=4326).geometry.iloc[nearest_indices].x.values,
    "distance_m": nearest_distances,
})

# save results to csv
nearest_df.to_csv("Data/fastfood_nearest_trafficlight.csv", index=False)

Non-point geometries detected — converting to centroids.


C:\Users\boie.2\AppData\Local\Temp\ipykernel_29892\2607581418.py:9: UserWarning:

Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.


C:\Users\boie.2\AppData\Local\Temp\ipykernel_29892\2607581418.py:39: UserWarning:

Geometry is in a geographic CRS. Results from 'distance' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.


C:\Users\boie.2\AppData\Local\Temp\ipykernel_29892\2607581418.py:39: UserWarning:

Geometry is in a geographic CRS. Results from 'distance' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.


C:\Users\boie.2\AppData\Local\Temp\ipykernel_29892\2607581418.py:39: UserWarning:

Geometry is in a geographic CRS. Results from 'distance' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS be

In [5]:
print(nearest_df[["fastfood_lat", "fastfood_lon"]].head())
print(nearest_df[["nearest_light_lat", "nearest_light_lon"]].head())

    fastfood_lat  fastfood_lon
18     39.994904    -83.006791
19     39.997545    -83.007422
22     39.987489    -83.028243
23     39.987559    -83.027769
24     40.021730    -83.013664
    nearest_light_lat  nearest_light_lon
18          39.994759         -83.007011
19          39.996827         -83.007433
22          39.987842         -83.025759
23          39.987842         -83.025759
24          40.021683         -83.013252


In [6]:
import plotly.express as px
import plotly.graph_objects as go

# build the map
fig = go.Figure()

fig.add_scattermapbox(
    lat=nearest_df["fastfood_lat"],
    lon=nearest_df["fastfood_lon"],
    mode="markers",
    marker=dict(size=8, color="blue"),
    name="Fast food"
)

fig.add_scattermapbox(
    lat=nearest_df["nearest_light_lat"],
    lon=nearest_df["nearest_light_lon"],
    mode="markers",
    marker=dict(size=6, color="red"),
    name="Traffic lights"
)

# Efficient connection lines
lats, lons = [], []
for _, row in nearest_df.iterrows():
    lats += [row["fastfood_lat"], row["nearest_light_lat"], None]
    lons += [row["fastfood_lon"], row["nearest_light_lon"], None]

fig.add_scattermapbox(
    lat=lats,
    lon=lons,
    mode="lines",
    line=dict(width=0.8, color="gray"),
    opacity=0.4,
    name="Connections"
)

fig.update_layout(
    mapbox_zoom=10,
    mapbox_center={"lat": 40.0, "lon": -83.0},
    title="Nearest Traffic Light Connections (Optimized)",
    margin=dict(r=0, t=40, l=0, b=0)
)
fig.update_layout(mapbox_style="carto-positron")
fig.show()
#fig.write_html("fastfood_trafficlight_connections.html")
#print("Saved map as fastfood_trafficlight_connections.html")


In [7]:
from geopy.distance import geodesic
def compute_geodesic_distance(row):
    coord1 = (row["fastfood_lat"], row["fastfood_lon"])
    coord2 = (row["nearest_light_lat"], row["nearest_light_lon"])
    return geodesic(coord1, coord2).meters

# Apply the distance calculation to each row
nearest_df["distance_m"] = nearest_df.apply(compute_geodesic_distance, axis=1)

In [8]:
nearest_df

,fastfood_name,fastfood_lat,fastfood_lon,nearest_light_lat,nearest_light_lon,distance_m
18,Raising Cane's,39.994904,-83.006791,39.994759,-83.007011,24.812774
19,Chipotle,39.997545,-83.007422,39.996827,-83.007433,79.706056
22,Jimmy John's,39.987489,-83.028243,39.987842,-83.025759,215.771575
23,Burger King,39.987559,-83.027769,39.987842,-83.025759,174.556150
24,Mama Mimi's Take 'n Bake Pizza,40.021730,-83.013664,40.021683,-83.013252,35.537669
...,...,...,...,...,...,...
5692,Subway,39.930148,-83.144734,39.931127,-83.144609,109.181721
5724,Taco Bell,40.018551,-83.025411,40.018027,-83.025945,73.894221
5725,Wendy's,40.035455,-83.016473,40.035163,-83.016807,43.236328
5740,Popeyes Louisiana Kitchen,40.123204,-83.089465,40.122175,-83.090200,130.369924


In [9]:

a = (39.9612, -82.9988)  # Columbus downtown
b = (39.9620, -82.9990)  # ~100 m away
print(geodesic(a, b).meters)

90.45588795908135


In [ ]:
# plot a distribution of distances, if the distance is more than 20 m, we assume that there is no light close
# to the drvie-thru exit

fig = px.histogram(
    nearest_df,
    x="distance_m",
    nbins=500,
    title="Distribution of restaurant distances to nearest traffic light",
    labels={"distance_m": "Distance (m)"},
    range_x=[0, 300]
)
fig.update_layout(bargap=0.05)
fig.show()

In [ ]:
# same plot for pizza in red

In [11]:
import osmnx as ox
import geopandas as gpd
G = ox.graph_from_place("Franklin County, Ohio, USA", network_type="drive")

In [12]:
import networkx as nx

def road_distance_m(lat1, lon1, lat2, lon2, G):
    # Find the nearest graph nodes to each coordinate
    orig_node = ox.distance.nearest_nodes(G, lon1, lat1)
    dest_node = ox.distance.nearest_nodes(G, lon2, lat2)
    
    # Compute the shortest path length in meters
    try:
        dist = nx.shortest_path_length(G, orig_node, dest_node, weight="length")
    except nx.NetworkXNoPath:
        dist = None  # no road connection found
    return dist

In [13]:
from tqdm import tqdm

tqdm.pandas()  # progress bar for apply

nearest_df["road_distance_m"] = nearest_df.progress_apply(
    lambda row: road_distance_m(
        row["fastfood_lat"],
        row["fastfood_lon"],
        row["nearest_light_lat"],
        row["nearest_light_lon"],
        G
    ),
    axis=1
)

100%|██████████| 792/792 [04:31<00:00,  2.92it/s]


In [15]:
fig = px.histogram(
    nearest_df,
    x="road_distance_m",
    nbins=1000,
    title="Distribution of road distances to nearest traffic light",
    labels={"distance_m": "Distance (m)"},
    range_x=[0, 300]
)
fig.update_layout(bargap=0.05)
fig.show()

In [14]:
fastfood_gdf

,name_normalized,amenity,geometry
18,Raising Cane's,fast_food,POINT (-83.00679 39.9949)
19,Chipotle,fast_food,POINT (-83.00742 39.99754)
22,Jimmy John's,fast_food,POINT (-83.02824 39.98749)
23,Burger King,fast_food,POINT (-83.02777 39.98756)
24,Mama Mimi's Take 'n Bake Pizza,fast_food,POINT (-83.01366 40.02173)
...,...,...,...
5692,Subway,fast_food,POINT (-83.14473 39.93015)
5724,Taco Bell,fast_food,POINT (-83.02541 40.01855)
5725,Wendy's,fast_food,POINT (-83.01647 40.03546)
5740,Popeyes Louisiana Kitchen,fast_food,POINT (-83.08946 40.1232)
